# CICCADA - Telemetry Data Quality Assessment

This notebook performs a structured data quality assessment of the battery fleet telemetry data (`all_data`).  
It assumes `all_data` has already been loaded as a single concatenated DataFrame with a `region` column.

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
import os
from dotenv import load_dotenv

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

In [ ]:
# Load data
# If the combined parquet data was saved earlier, use this:


load_dotenv()
BASE = Path(os.environ["BATTERY_DATA_ROOT"])
all_data = pd.read_parquet(BASE / 'combined_telemetry.parquet')

## 2. Schema & Basic Statistics

In [ ]:
# Dtypes
print("=== Dtypes ===")
print(all_data.dtypes)

print("\n=== Descriptive statistics (numeric columns) ===")
all_data.describe().T

In [ ]:
# Global null counts and rates
null_counts = all_data.isnull().sum()
null_rates  = (null_counts / len(all_data) * 100).round(2)

null_summary = pd.DataFrame({
    'null_count': null_counts,
    'null_pct':   null_rates
}).sort_values('null_pct', ascending=False)

print("=== Global Null Summary ===")
print(null_summary.to_string())

# Visual
fig, ax = plt.subplots(figsize=(10, 5))
null_summary['null_pct'].plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Null rate (%)')
ax.set_title('Null rate per column (all sites combined)')
ax.axvline(1,  color='orange', linestyle='--', label='1% threshold')
ax.axvline(10, color='red',    linestyle='--', label='10% threshold')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Temporal Integrity
Data is expected at exactly 1-minute (60-second) intervals per site.  
Checking for missing rows, duplicate timestamps, and non-60s gaps.

In [ ]:
# Expected date range
ts_min = all_data['timestamp'].min()
ts_max = all_data['timestamp'].max()
expected_index = pd.date_range(ts_min, ts_max, freq='1min')
expected_rows_per_site = len(expected_index)
print(f"Expected rows per site (if no gaps): {expected_rows_per_site:,}")

In [ ]:
# Check duplicate timestamps per site
dupes = (
    all_data
    .groupby('pseudonym')
    .apply(lambda g: g.duplicated(subset='timestamp').sum())
    .rename('duplicate_timestamps')
)
print("=== Duplicate timestamps per site ===")
print(dupes[dupes > 0].to_string() if (dupes > 0).any() else "No duplicates found.")

In [ ]:
# Time delta distribution
# Compute per-site time deltas
all_data['dt_seconds'] = (
    all_data
    .groupby('pseudonym')['timestamp']
    .diff()
    .dt.total_seconds()
)

dt_counts = all_data['dt_seconds'].value_counts(dropna=True).head(15)
print("=== Top 15 time delta values (seconds) ===")
print(dt_counts.to_string())

# Pct of intervals that are exactly 60s
non_null_deltas = all_data['dt_seconds'].dropna()
pct_60s = (non_null_deltas == 60).mean() * 100
print(f"\nIntervals exactly 60s: {pct_60s:.4f}%")

In [ ]:
# Per-site: actual row count vs expected, and missing row count
site_row_counts = all_data.groupby('pseudonym').size().rename('actual_rows')
site_temporal = site_row_counts.to_frame()
site_temporal['expected_rows'] = expected_rows_per_site
site_temporal['missing_rows']  = site_temporal['expected_rows'] - site_temporal['actual_rows']
site_temporal['completeness_pct'] = (site_temporal['actual_rows'] / site_temporal['expected_rows'] * 100).round(2)

print("=== Sites with <99% temporal completeness ===")
low_completeness = site_temporal[site_temporal['completeness_pct'] < 99]
print(low_completeness.sort_values('completeness_pct').to_string() if len(low_completeness) else "All sites ≥99% complete.")

# Histogram
fig, ax = plt.subplots(figsize=(10, 4))
site_temporal['completeness_pct'].hist(bins=30, ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Completeness (%)')
ax.set_ylabel('Number of sites')
ax.set_title('Temporal completeness per site (1-min resolution)')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.transforms as mtransforms

GAP_THRESHOLD = pd.Timedelta('2min')

ts_by_site = {
    site: grp.sort_values().reset_index(drop=True)
    for site, grp in all_data.groupby('pseudonym', observed=True)['timestamp']
}

# Sort SN001 → SN100 by the numeric suffix
sites_ordered = sorted(site_temporal.index.tolist(), key=lambda s: int(s[2:]))
n_sites = len(sites_ordered)

global_start = all_data['timestamp'].min()
global_end   = all_data['timestamp'].max()

CLUSTERS = [
    ('SN001–025',  0, 24, '#fff3cd'),   # amber
    ('SN026–050', 25, 49, '#cfe2ff'),   # blue
    ('SN051–100', 50, 99, '#d1e7dd'),   # green
]

fig, ax = plt.subplots(figsize=(18, 20))

# Cluster background bands (drawn first, behind everything)
for _label, start, end, color in CLUSTERS:
    ax.axhspan(start - 0.5, end + 0.5, color=color, alpha=0.35, zorder=0)

for i, site in enumerate(sites_ordered):
    site_ts = ts_by_site[site]
    site_start = site_ts.iloc[0]
    site_end   = site_ts.iloc[-1]

    ax.barh(i, site_end - site_start, left=site_start,
            height=0.8, color='#d0d0d0', linewidth=0, zorder=1)

    diffs = site_ts.diff()
    gap_positions = diffs[diffs > GAP_THRESHOLD].index.tolist()

    block_start_ts = site_start
    for gp in gap_positions:
        ax.barh(i, site_ts.iloc[gp - 1] - block_start_ts, left=block_start_ts,
                height=0.8, color='steelblue', linewidth=0, zorder=2)
        block_start_ts = site_ts.iloc[gp]
    ax.barh(i, site_end - block_start_ts, left=block_start_ts,
            height=0.8, color='steelblue', linewidth=0, zorder=2)

# Cluster dividers + right-side labels (blended: x in axes fraction, y in data coords)
trans = mtransforms.blended_transform_factory(ax.transAxes, ax.transData)
for label, start, end, _color in CLUSTERS:
    if end < n_sites - 1:
        ax.axhline(end + 0.5, color='#666', linewidth=1.2, linestyle='--', zorder=3)
    ax.text(1.01, (start + end) / 2, label, transform=trans,
            va='center', ha='left', fontsize=9, fontweight='bold', color='#333')

ax.set_xlim(global_start, global_end)
ax.set_ylim(-0.5, n_sites - 0.5)
ax.invert_yaxis()   # SN001 at top
ax.set_yticks(range(n_sites))
ax.set_yticklabels(
    [f"{s}  ({site_temporal.loc[s, 'completeness_pct']:.1f}%)" for s in sites_ordered],
    fontsize=7
)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
fig.autofmt_xdate(rotation=45)
ax.set_xlabel('Date')
ax.set_title(
    'Temporal coverage per site',
    fontsize=11
)
ax.legend(
    handles=[
        mpatches.Patch(color='steelblue', label='Data present'),
        mpatches.Patch(color='#d0d0d0',   label='Gap within site span'),
    ],
    loc='lower right', framealpha=0.9
)
plt.tight_layout()
plt.show()


In [ ]:
# Large gaps: find intervals > 60 min
large_gaps = all_data[all_data['dt_seconds'] > 3600][['pseudonym', 'timestamp', 'dt_seconds']].copy()
large_gaps['gap_hours'] = (large_gaps['dt_seconds'] / 3600).round(1)
print(f"=== Large gaps (>1 hour): {len(large_gaps)} instances across {large_gaps['pseudonym'].nunique()} sites ===")
print(large_gaps.sort_values('gap_hours', ascending=False).head(20).to_string())

## 4. Column-by-Column Quality Assessment

In [ ]:
# See all columns below:
all_data.columns

### 4.1 `grid_connected_status`

**Expected (per manual - page 64, table 8):** Text — `"Grid Connected"` or `"Grid Disconnected"`  
**Observed:** Float values ~49.97 to 50.02  
**Hypothesis:** This column is potentially **grid frequency (Hz)**, mislabelled or incorrectly mapped in the data pipeline. 
The expected true grid status signal should be `grid_connected_status` (text), but what is present resembles `grid_frequency` (Hz).

In [ ]:
col = 'grid_connected_status'

print(f"dtype: {all_data[col].dtype}")
print(f"Null count: {all_data[col].isnull().sum():,}")
print(f"\nDescriptive stats:")
print(all_data[col].describe())

print(f"\nSample unique values (first 20 sorted):")
print(sorted(all_data[col].dropna().unique())[:20])

# Visualise distribution
fig, ax = plt.subplots(figsize=(10, 4))
all_data[col].dropna().hist(bins=100, ax=ax, color='steelblue', edgecolor='white')
ax.axvline(49.75, color='red',    linestyle='--', label='f_LLCO (49.75 Hz, Table 4.4 Aus A)')
ax.axvline(50.25, color='orange', linestyle='--', label='f_ULCO (50.25 Hz, Table 4.4 Aus A)')
ax.set_xlabel('Value')
ax.set_title(f'"{col}" distribution (expected: text; observed: float resembling Hz)')
ax.legend()
plt.tight_layout()
plt.show()

# If it IS frequency: how often is it out of the AS/NZS 4777.2 normal operating band (49–51 Hz)?
# Check Table 4.4. from AS/NZS 4777.2:2020 (page 34)
if all_data[col].between(45, 55).mean() > 0.99:
    out_of_band = all_data[col].dropna()
    pct_below = (out_of_band < 49.75).mean() * 100
    pct_above = (out_of_band > 50.25).mean() * 100
    print(f"\n[If treated as Hz] Values <49.75 Hz (below f_LLCO, Table 4.4 Aus A): {pct_below:.3f}%")
    print(f"[If treated as Hz] Values >50.25 Hz (above f_ULCO, Table 4.4 Aus A): {pct_above:.3f}%")
    # Trip thresholds — Table 4.2 (Passive anti-islanding frequency limits, Aus A)
    print(f"[If treated as Hz] Values <47 Hz (trip zone, Table 4.2 Aus A): {(out_of_band < 47.0).mean()*100:.4f}%")
    print(f"[If treated as Hz] Values >52 Hz (trip zone, Table 4.2 Aus A): {(out_of_band > 52.0).mean()*100:.4f}%")

### 4.2 `AC_voltage`

**Expected:** AC voltage at inverter terminals (V)  
**Nominal:** 230 V (single-phase Australian standard)  
**Plausible range per AS/NZS 4777.2 (Australia A defaults):**
- **VV1 = 207 V** - lower Volt-VAr onset (reactive power supply begins)
- **VV2 = 220 V** - lower Volt-VAr deadband edge (reactive power → 0)
- **VV3 = 240 V** - upper Volt-VAr deadband edge (reactive power absorption begins)
- **VW1 = 253 V** - Volt-Watt onset (active power curtailment starts here)
- **VV4 = 258 V** - upper Volt-VAr onset (full reactive absorption)
- **VW2 = 260 V** - Volt-Watt ramp end (output capped at 20% of rated above here)

In [ ]:
col = 'AC_voltage'

print(all_data[col].describe())
print(f"\nNull count: {all_data[col].isnull().sum():,}")

v = all_data[col].dropna()

# Threshold analysis (using Australia A for now)
thresholds = [
    (v < 180,            'Below 180 V (below Volt-VAr V_v1 allowed range)'),
    (v.between(180, 207), '180-207 V (Volt-VAr reactive supply zone, below V_v1)'),
    (v.between(207, 220), '207-220 V (Volt-VAr transition - supplying, V_v1 to V_v2)'),
    (v.between(220, 240), '220-240 V (normal/unity PF zone, V_v2 to V_v3)'),
    (v.between(240, 253), '240-253 V (Volt-VAr absorbing zone, V_v3 to V_w1)'),
    (v.between(253, 260), '253-260 V (Volt-Watt active ramp down, V_w1 to V_w2)'),
    (v > 260,            'Above 260 V (above V_w2 - output capped at 20%)'),
]
print("\n=== Voltage band breakdown ===")
for mask, label in thresholds:
    print(f"  {label}: {mask.sum():,} rows ({mask.mean()*100:.3f}%)")

# Distribution
fig, ax = plt.subplots(figsize=(12, 6))
v.hist(bins=200, ax=ax, color='steelblue', edgecolor='none')

for xval, label, colour in [
    (207, 'V_v1 (207V)',  'blue'),
    (220, 'V_v2 (220V)',  'cyan'),
    (240, 'V_v3 (240V)',  'orange'),
    (253, 'V_w1 (253V)',  'red'),
    (258, 'V_v4 (258V)',  'purple'),
    (260, 'V_w2 (260V)',  'darkred'),
]:
    ax.axvline(xval, linestyle='--', color=colour, label=label)

ax.set_xlim(180, 300)
ax.set_xlabel('AC Voltage (V)')
ax.set_title('AC_voltage distribution across all sites')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


# Per-site: % of time in Volt-Watt zone
voltwatt_pct = (
    all_data.dropna(subset=[col])
    .groupby('pseudonym')[col]
    .apply(lambda x: x.between(253, 260).mean() * 100)
    .rename('pct_time_voltwatt_zone')
    .sort_values(ascending=False)
)
#print("\n=== Sites: % time in Volt-Watt active ramp (V_w1-V_w2, 253-260 V) ===")
#print(voltwatt_pct.to_string())

In [ ]:
# Frozen/stuck voltage detection (same value for ≥5 consecutive minutes)
all_data['voltage_frozen'] = (
    all_data.groupby('pseudonym')[col]
    .transform(lambda x: (x.diff() == 0).rolling(5, min_periods=5).sum() == 5)
)
frozen_pct = all_data['voltage_frozen'].mean() * 100
frozen_by_site = all_data.groupby('pseudonym')['voltage_frozen'].mean().sort_values(ascending=False) * 100
print(f"Overall frozen voltage rate: {frozen_pct:.3f}%")
print("\nTop 10 sites by frozen voltage rate (%)")
print(frozen_by_site.head(10).to_string())

### 4.3 `solar_instant_power`

**Expected:** Instantaneous solar real power (W), should be ≥0 (generation only).  
**Quality checks:** negative values, nighttime solar, implausibly high values, nulls.

In [ ]:
col = 'solar_instant_power'
print(all_data[col].describe())
print(f"\nNull count: {all_data[col].isnull().sum():,}")

s = all_data[col].dropna()

# Negative values (metering error per manual: fails if solar < -250W > 1% of time)
# See powerhub manual page 14
neg_mask = s < -250
print(f"\nValues < -250W (manual 'Negative Solar' diagnostic threshold): {neg_mask.sum():,} ({neg_mask.mean()*100:.3f}%)")

# Any negative at all
any_neg = s < 0
print(f"Values < 0W (any negative): {any_neg.sum():,} ({any_neg.mean()*100:.3f}%)")

# Per-site negative solar rate
neg_by_site = (
    all_data.dropna(subset=[col])
    .groupby('pseudonym')[col]
    .apply(lambda x: (x < -250).mean() * 100)
    .rename('pct_negative_solar')
    .sort_values(ascending=False)
)
print("\nSites with >1% negative solar (would fail Powerhub diagnostic):")
failing = neg_by_site[neg_by_site > 1]
print(failing.to_string() if len(failing) else "None")

# Distribution (excluding zeros for clarity)
fig, ax = plt.subplots(figsize=(10, 4))
s[s != 0].hist(bins=200, ax=ax, color='gold', edgecolor='none')
ax.axvline(0, color='red', linestyle='--', label='Zero')
ax.set_xlabel('solar_instant_power (W)')
ax.set_title('Solar instant power distribution (non-zero values)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
slice = all_data[all_data['timestamp'].between('2026-01-20', '2026-01-26') & (all_data['pseudonym'] == 'SN001')]

In [ ]:
slice.plot(x='timestamp', y='solar_instant_power', figsize=(24, 4))

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

df = all_data[all_data['pseudonym'] == 'SN001'].sort_values('timestamp')

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(x=df['timestamp'], y=df['solar_instant_power'],
               name='Solar power (W)', line=dict(color='goldenrod', width=1)),
    secondary_y=False
)

fig.add_trace(
    go.Scatter(x=df['timestamp'], y=df['AC_voltage'],
               name='AC voltage (V)', line=dict(color='steelblue', width=1)),
    secondary_y=True
)

fig.update_yaxes(title_text='Solar power (W)', secondary_y=False)
fig.update_yaxes(title_text='AC voltage (V)', secondary_y=True)

fig.update_layout(
    title='SN042 — solar_instant_power & AC_voltage',
    height=400, width=1400,
    hovermode='x unified',
    template='plotly_white'
)

fig.update_xaxes(
    rangeslider_visible=True,
    rangeselector=dict(
        buttons=[
            dict(count=1, label='1d', step='day', stepmode='backward'),
            dict(count=7, label='7d', step='day', stepmode='backward'),
            dict(step='all')
        ]
    )
)

fig.show()


### 4.4 `solar_real_power_limit`

**Expected:** Maximum allowed solar output (W). This potentially is the **curtailment ceiling**.  
When `solar_instant_power < solar_real_power_limit`, the inverter is operating below capacity.  
High null rate (~80% in SN026). 
Important to understand whether nulls mean 'no limit active' or 'data unavailable'.

In [ ]:
col = 'solar_real_power_limit'
print(all_data[col].describe())
print(f"\nNull count: {all_data[col].isnull().sum():,} ({all_data[col].isnull().mean()*100:.1f}%)")

# Null rate per site
null_by_site = (
    all_data.groupby('pseudonym')[col]
    .apply(lambda x: x.isnull().mean() * 100)
    .rename('pct_null')
)
print("\n=== Null rate distribution across sites ===")
print(null_by_site.describe())

fig, ax = plt.subplots(figsize=(10, 4))
null_by_site.hist(bins=30, ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Null rate per site (%)')
ax.set_title(f'Per-site null rate: {col}')
plt.tight_layout()
plt.show()

# When non-null: does it equal zero? (zero limit = full curtailment command)
non_null = all_data[col].dropna()
print(f"\nNon-null values that are exactly 0W (full curtailment): {(non_null == 0).sum():,} ({(non_null == 0).mean()*100:.2f}%)")

# Curtailment proxy: rows where solar_instant_power >= solar_real_power_limit - 50W
# (i.e. actively hitting the limit)
both_present = all_data.dropna(subset=[col, 'solar_instant_power'])
curtailed = (both_present['solar_instant_power'] >= both_present[col] - 50)
print(f"\nRows where solar_instant_power ≈ solar_real_power_limit (within 50W): {curtailed.sum():,} ({curtailed.mean()*100:.2f}% of rows with both values present)")

### 4.5 `battery_inverter_real_power` and `battery_target_power`

**Sign convention (per manual):** Positive = discharge, Negative = charge  
**Quality checks:** nulls, plausibility, how closely does actual power track the target command?

In [ ]:
# See powerhub manual page 67 for battery_inverter_real_power, and page 62 for battery_target_power.

for col in ['battery_inverter_real_power', 'battery_target_power']:
    s = all_data[col].dropna()
    print(f"\n=== {col} ===")
    print(s.describe())
    print(f"Null: {all_data[col].isnull().sum():,} ({all_data[col].isnull().mean()*100:.2f}%)")
    print(f"Charging (< 0): {(s < 0).sum():,} ({(s < 0).mean()*100:.1f}%)")
    print(f"Discharging (> 0): {(s > 0).sum():,} ({(s > 0).mean()*100:.1f}%)")
    print(f"Idle (== 0): {(s == 0).sum():,} ({(s == 0).mean()*100:.1f}%)")

# Tracking error: actual vs target
both = all_data.dropna(subset=['battery_inverter_real_power', 'battery_target_power'])
both = both.assign(
    tracking_error = both['battery_inverter_real_power'] - both['battery_target_power']
)
print("\n=== Tracking error (actual - target) ===")
print(both['tracking_error'].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, col in zip(axes, ['battery_inverter_real_power', 'battery_target_power']):
    all_data[col].dropna().hist(bins=200, ax=ax, color='steelblue', edgecolor='none')
    ax.axvline(0, color='red', linestyle='--')
    ax.set_xlabel('Power (W)')
    ax.set_title(col)
plt.suptitle('Battery power distribution (positive=discharge, negative=charge)')
plt.tight_layout()
plt.show()

### 4.6 `solar_reactive_power` and `battery_inverter_reactive_power`

**Quality checks:** nulls, range.
Per AS/NZS 4777.2 Table 3.7 (Australia A defaults), Volt-VAr response caps reactive power at:
- **44% of rated apparent power supplying** (at VV1 = 207 V and below)
- **60% of rated apparent power absorbing** (at VV4 = 258 V and above)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, col in zip(axes, ['solar_reactive_power', 'battery_inverter_reactive_power']):
    s = all_data[col].dropna()
    s.hist(bins=100, ax=ax, color='coral', edgecolor='none')
    ax.axvline(0, color='black', linestyle='--')
    ax.set_xlabel('Reactive Power (VAr)')
    ax.set_title(col)
    print(f"\n=== {col} ===")
    print(s.describe())
    print(f"Null: {all_data[col].isnull().sum():,} ({all_data[col].isnull().mean()*100:.2f}%)")
plt.tight_layout()
plt.show()

### 4.7 `site_energy_exported` and `site_energy_imported`

**Expected:** Cumulative energy (Wh). These seem to be **running totals** (odometer-style), not interval energy.

**Quality checks:** non-monotonic (counter rollover or reset), negative values, nulls, implausible jumps.

In [ ]:
all_data[all_data['pseudonym'] == 'SN005'].plot(x='timestamp', y='site_energy_exported', figsize=(24,6))

In [ ]:
slice.plot(x='timestamp', y='site_energy_exported',figsize=(12, 4))

In [ ]:
for col in ['site_energy_exported', 'site_energy_imported']:
    print(f"\n=== {col} ===")
    print(all_data[col].describe())
    print(f"Null: {all_data[col].isnull().sum():,} ({all_data[col].isnull().mean()*100:.2f}%)")
    print(f"Negative values: {(all_data[col] < 0).sum():,}")

    # Non-monotonic check: per-site, compute diffs.
    # Negative diff = counter reset or error
    diffs = (
        all_data.dropna(subset=[col])
        .groupby('pseudonym')[col]
        .diff()
    )
    n_decreases = (diffs < -100).sum()  # allow small floating point noise
    n_big_jumps = (diffs > 1e6).sum()   # >1 MWh in one minute = likely error
    print(f"Non-monotonic drops (diff < -100 Wh): {n_decreases:,}")
    print(f"Implausible jumps (diff > 1 MWh/min): {n_big_jumps:,}")

### 4.8 `solar_to_battery_energy` and `solar_to_load_energy`

**Expected:** Energy flows (Wh). Should be ≥ 0  

In [ ]:
for col in ['solar_to_battery_energy', 'solar_to_load_energy']:
    print(f"\n=== {col} ===")
    null_by_site = all_data.groupby('pseudonym')[col].apply(lambda x: x.isnull().mean() * 100)
    print(f"Overall null rate: {all_data[col].isnull().mean()*100:.1f}%")
    print(f"Sites with 100% nulls: {(null_by_site == 100).sum()}")
    print(f"Sites with 0% nulls:   {(null_by_site == 0).sum()}")
    print(f"Sites with partial nulls: {((null_by_site > 0) & (null_by_site < 100)).sum()}")

    non_null = all_data[col].dropna()
    if len(non_null):
        print(f"Non-null describe:")
        print(non_null.describe())
        print(f"Negative values: {(non_null < 0).sum():,}")

# Are the sites with data clustered by region?
print("\n=== solar_to_battery_energy: null rate by region ===")
if 'region' in all_data.columns:
    print(all_data.groupby('region')['solar_to_battery_energy'].apply(lambda x: x.isnull().mean() * 100).rename('null_pct'))

In [ ]:
import matplotlib.transforms as mtransforms

cols = ['solar_to_battery_energy', 'solar_to_load_energy']
short_names = {'solar_to_battery_energy': 'solar→battery',
               'solar_to_load_energy':    'solar→load'}

# Per-site coverage (% non-null), sorted SN001→SN100
coverage = pd.DataFrame({
    col: all_data.groupby('pseudonym', observed=True)[col]
                 .apply(lambda x: x.notna().mean() * 100).round(1)
    for col in cols
})
coverage = coverage.loc[sorted(coverage.index, key=lambda s: int(s[2:]))]

# Printed summary
for col in cols:
    v = coverage[col]
    print(f"\n{col}:")
    print(f"  Full (100%):      {(v == 100).sum():3d} sites")
    print(f"  Partial (0–100%): {((v > 0) & (v < 100)).sum():3d} sites")
    print(f"  None (0%):        {(v == 0).sum():3d} sites")

partial_mask = (
    ((coverage[cols[0]] > 0) & (coverage[cols[0]] < 100)) |
    ((coverage[cols[1]] > 0) & (coverage[cols[1]] < 100))
)
if partial_mask.any():
    print("\n=== Sites with partial coverage ===")
    print(coverage[partial_mask].to_string())

# Heatmap 
CLUSTERS = [
    ('SN001–025',  0,  25),   # heatmap rows 0–24,  divider at y=25
    ('SN026–050', 25,  50),   # heatmap rows 25–49, divider at y=50
    ('SN051–100', 50, 100),   # heatmap rows 50–99
]

fig, ax = plt.subplots(figsize=(5, 18))
sns.heatmap(
    coverage.rename(columns=short_names),
    ax=ax,
    cmap='RdYlGn',
    vmin=0, vmax=100,
    linewidths=0.4, linecolor='white',
    cbar_kws={'label': 'Coverage (% non-null)', 'shrink': 0.4},
    annot=True, fmt='.0f', annot_kws={'size': 6.5}
)

trans = mtransforms.blended_transform_factory(ax.transAxes, ax.transData)
for i, (label, start, end) in enumerate(CLUSTERS):
    if i < len(CLUSTERS) - 1:
        ax.axhline(end, color='#222', linewidth=2, zorder=5)
    ax.text(-0.12, (start + end) / 2, label, transform=trans,
            va='center', ha='right', fontsize=8.5, fontweight='bold', color='#333')

ax.set_title(
    'Per-site coverage: solar energy flow columns\n(% of rows with non-null values)',
    pad=10, fontsize=11
)
ax.set_xlabel('')
plt.tight_layout()
plt.show()


In [ ]:
df = all_data[all_data['pseudonym'] == 'SN026']

In [ ]:
df[['timestamp'] + cols].dropna().plot(
    x='timestamp', y=cols, figsize=(24, 4),
    drawstyle='steps-post'
)


In [ ]:
import plotly.graph_objects as go

df_5min = df[['timestamp'] + cols].dropna()

fig = go.Figure()
for col in cols:
    fig.add_trace(go.Scatter(
        x=df_5min['timestamp'],
        y=df_5min[col],
        name=col,
        line_shape='hv'
    ))

fig.update_layout(
    height=400, width=1400,
    hovermode='x unified',
    template='plotly_white'
)
fig.update_xaxes(
    rangeslider_visible=True,
    rangeselector=dict(
        buttons=[
            dict(count=1, label='1d', step='day', stepmode='backward'),
            dict(count=7, label='7d', step='day', stepmode='backward'),
            dict(step='all')
        ]
    )
)
fig.show()


### 4.9 `battery_target_reactive_power`

**Expected:** Reactive power command from Site Controller (VAr)  
**Quality checks:** nulls, whether it is ever non-zero (active Volt-VAr dispatch)

In [ ]:
col = 'battery_target_reactive_power'
print(all_data[col].describe())
print(f"Null: {all_data[col].isnull().sum():,} ({all_data[col].isnull().mean()*100:.2f}%)")
print(f"Exactly zero: {(all_data[col] == 0).sum():,} ({(all_data[col] == 0).mean()*100:.2f}%)")
nonzero = all_data[col].dropna()
nonzero = nonzero[nonzero != 0]
print(f"Non-zero values: {len(nonzero):,}")
if len(nonzero):
    print(nonzero.describe())

### 4.10 `csip_opmod_exp_lim`

**Expected:** CSIP-AUS export limit (W) active on this site.
This is the export limit dispatched via the Common Smart Inverter Profile (Australia).  
When non-null and non-zero, an export limit is active.  
When zero, full curtailment has been commanded.  
**When null:** the CSIP-AUS connection may not be active, or the field was not transmitted.
**Note:** Can't find this parameter in the powerhub manual

In [ ]:
col = 'csip_opmod_exp_lim'
print(all_data[col].describe())
print(f"\nNull: {all_data[col].isnull().sum():,} ({all_data[col].isnull().mean()*100:.2f}%)")
print(f"Zero (full curtailment commanded): {(all_data[col] == 0).sum():,}")

non_null = all_data[col].dropna()
print(f"\nValue counts (top 20):")
print(non_null.value_counts().head(20).to_string())

# Null rate per site
null_by_site = all_data.groupby('pseudonym')[col].apply(lambda x: x.isnull().mean() * 100).rename('null_pct')
print("\n=== Null rate distribution across sites ===")
print(null_by_site.describe())

# Are nulls clustered in time? (suggests CSIP not yet deployed at start of dataset)
all_data['csip_null'] = all_data[col].isnull()
csip_null_over_time = (
    all_data.set_index('timestamp')
    .resample('1D')['csip_null']
    .mean() * 100
)
fig, ax = plt.subplots(figsize=(12, 4))
csip_null_over_time.plot(ax=ax, color='steelblue')
ax.set_ylabel('Null rate (%)')
ax.set_title('csip_opmod_exp_lim daily null rate (all sites)')
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

# When non-null and > 0: distribution of limit values
active_limits = non_null[non_null > 0]
if len(active_limits):
    print(f"\nActive export limits (>0W): {len(active_limits):,} rows")
    print(active_limits.describe())

In [ ]:
import plotly.graph_objects as go

col = 'csip_opmod_exp_lim'

# Daily mean per site — reduces 42M rows to ~100 sites × ~460 days
daily = (
    all_data[['pseudonym', 'timestamp', col]]
    .set_index('timestamp')
    .groupby('pseudonym', observed=True)[col]
    .resample('1D').mean()
    .reset_index()
)

pivot = (
    daily.pivot(index='pseudonym', columns='timestamp', values=col)
         .loc[sorted(daily['pseudonym'].unique(), key=lambda s: int(s[2:]))]
)

sites = pivot.index.tolist()
dates = pivot.columns.tolist()

CLUSTERS = [
    ('SN001–025',  0, 24, 24.5),
    ('SN026–050', 25, 49, 49.5),
    ('SN051–100', 50, 99, None),
]

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=dates,
    y=sites,
    colorscale='Blues',
    zmin=0,
    xgap=0, ygap=0,
    hoverongaps=False,
    colorbar=dict(title='Daily mean<br>CSIP limit (W)'),
))

# Cluster dividers
for _label, _start, _end, boundary in CLUSTERS:
    if boundary is not None:
        fig.add_shape(
            type='line', xref='paper', yref='y',
            x0=0, x1=1, y0=boundary, y1=boundary,
            line=dict(color='tomato', width=2, dash='dot')
        )

'''# Cluster labels in right margin
for label, start, end, _boundary in CLUSTERS:
    fig.add_annotation(
        xref='paper', yref='y',
        x=1.01, y=(start + end) / 2,
        text=label, showarrow=False,
        xanchor='left', font=dict(size=10, color='#333')
    )
'''
fig.update_layout(
    title=(
        'csip_opmod_exp_lim per site — daily mean<br>'
        '<sup>Grey = null (not deployed / signal absent) | '
        'White = 0 W (full curtailment) | Blue = active export limit</sup>'
    ),
    height=800, width=800,
    yaxis=dict(autorange='reversed'),  # SN001 at top
    plot_bgcolor='#bbbbbb',            # NaN cells are transparent → show grey here
    template='plotly_white',
    margin=dict(r=130),
)
fig.show()


## 5. Cross-Column Physical Plausibility
These checks test whether combinations of columns make physical sense.

In [ ]:
# Check 1: Battery discharging AND exporting > limit
# If csip_opmod_exp_lim is active, site_energy_exported shouldn't be growing
# faster than the limit permits. This is a rough proxy — exact check needs
# interval energy, not cumulative.
# Here we flag rows where battery is discharging (>0) and csip limit is 0
# (should mean no export allowed).
check1 = all_data[
    (all_data['battery_inverter_real_power'] > 100) &  # battery discharging
    (all_data['csip_opmod_exp_lim'] == 0)              # zero export limit active
]
print(f"Check 1 — Battery discharging while CSIP export limit = 0: {len(check1):,} rows ({len(check1)/len(all_data)*100:.2f}%)")
# Note: this isn't necessarily wrong (battery could be serving load, not exporting)
# but worth reviewing.

In [ ]:
# Check 2: Solar exceeds its own limit
# solar_instant_power should not significantly exceed solar_real_power_limit
both = all_data.dropna(subset=['solar_instant_power', 'solar_real_power_limit'])
over_limit = both[both['solar_instant_power'] > both['solar_real_power_limit'] + 100]  # 100W tolerance
print(f"Check 2 — solar_instant_power > solar_real_power_limit + 100W: {len(over_limit):,} rows ({len(over_limit)/len(both)*100:.2f}% of rows with both present)")
if len(over_limit):
    print("Example rows:")
    print(over_limit[['pseudonym', 'timestamp', 'solar_instant_power', 'solar_real_power_limit']].head(10).to_string())

In [ ]:
# Check 3: Negative home load proxy
# The Powerhub diagnostic 'Negative Home Load' fires if:
# load_real_power < -100W more than 1% of the time.
# We don't have load_real_power directly, but we can approximate:
# Approx load = solar - battery_discharge + grid_import
# (This is only possible if the sign convention is consistent)
# For now, flag if solar_to_load_energy goes negative
neg_stl = all_data[all_data['solar_to_load_energy'] < 0]
print(f"Check 3 - solar_to_load_energy < 0: {len(neg_stl):,} rows")

In [ ]:
# Check 4: Target vs actual reactive power consistency
# Reactive power target is almost always 0 — flag sites where
# actual battery_inverter_reactive_power is large when target is 0
both_rp = all_data.dropna(subset=['battery_inverter_reactive_power', 'battery_target_reactive_power'])
unexpected_reactive = both_rp[
    (both_rp['battery_target_reactive_power'] == 0) &
    (both_rp['battery_inverter_reactive_power'].abs() > 500)
]
print(f"Check 4 — Large reactive power when target is 0 (>500 VAr): {len(unexpected_reactive):,} rows")
if len(unexpected_reactive):
    print(unexpected_reactive.groupby('pseudonym').size().sort_values(ascending=False).head(10))

## 6. Per-Site Quality Summary

All quality dimensions into a single per-site summary table.

In [ ]:
numeric_cols = [
    'grid_connected_status', 'solar_reactive_power', 'battery_inverter_reactive_power',
    'battery_inverter_real_power', 'AC_voltage', 'battery_target_power',
    'solar_instant_power', 'solar_real_power_limit', 'site_energy_exported',
    'site_energy_imported', 'solar_to_battery_energy', 'solar_to_load_energy',
    'battery_target_reactive_power', 'csip_opmod_exp_lim'
]

def site_quality(g):
    n = len(g)
    row = {}
    row['n_rows'] = n
    row['completeness_pct'] = round(n / expected_rows_per_site * 100, 2)
    row['duplicate_ts'] = g.duplicated(subset='timestamp').sum()
    row['n_large_gaps'] = (g['dt_seconds'] > 3600).sum()

    for col in numeric_cols:
        null_rate = g[col].isnull().mean() * 100
        row[f'{col}_null_pct'] = round(null_rate, 1)

    # Key derived flags
    row['pct_voltwatt_zone']   = round(g['AC_voltage'].between(253, 260).mean() * 100, 2)  # VW1-VW2 ramp
    row['pct_above_VW1']       = round((g['AC_voltage'] > 253).mean() * 100, 2)            # any VW exposure
    row['pct_above_VW2']       = round((g['AC_voltage'] > 260).mean() * 100, 2)            
    row['pct_negative_solar']  = round((g['solar_instant_power'] < -250).mean() * 100, 3)
    row['csip_active_rows']    = (g['csip_opmod_exp_lim'].notna() & (g['csip_opmod_exp_lim'] >= 0)).sum()
    row['csip_zero_limit_rows'] = (g['csip_opmod_exp_lim'] == 0).sum()

    return pd.Series(row)

print("Computing per-site quality summary...")
site_quality_df = all_data.groupby('pseudonym').apply(site_quality)

if 'region' in all_data.columns:
    site_quality_df = site_quality_df.join(
        all_data.groupby('pseudonym')['region'].first()
    )

print("Done.")
print(f"\nQuality summary shape: {site_quality_df.shape}")
site_quality_df

In [ ]:
# Heatmap: null rates across columns and sites
null_cols = [c for c in site_quality_df.columns if c.endswith('_null_pct')]
null_heat = site_quality_df[null_cols].copy()
null_heat.columns = [c.replace('_null_pct', '') for c in null_cols]

fig, ax = plt.subplots(figsize=(16, max(8, len(null_heat) * 0.18)))
sns.heatmap(
    null_heat,
    ax=ax,
    cmap='YlOrRd',
    vmin=0, vmax=100,
    linewidths=0.1,
    cbar_kws={'label': 'Null rate (%)'}
)
ax.set_title('Per-site null rate heatmap (each row = one battery, each column = one signal)')
ax.set_xlabel('Signal')
ax.set_ylabel('Site (pseudonym)')
plt.tight_layout()
plt.show()

In [ ]:
# Summary stats by region
if 'region' in site_quality_df.columns:
    print("=== Mean completeness and key null rates by region ===")
    key_cols = ['completeness_pct', 'csip_opmod_exp_lim_null_pct',
                'solar_real_power_limit_null_pct', 'pct_voltwatt_zone',
                'pct_above_VW1', 'pct_above_VW2']
    key_cols = [c for c in key_cols if c in site_quality_df.columns]
    print(site_quality_df.groupby('region')[key_cols].mean().round(2).to_string())

## 7. Flags & Export

In [ ]:
# Assign pass/flag status on key dimensions
flags = site_quality_df[['completeness_pct']].copy()
if 'region' in site_quality_df.columns:
    flags['region'] = site_quality_df['region']

flags['FLAG_low_completeness']       = site_quality_df['completeness_pct'] < 95
flags['FLAG_has_duplicates']         = site_quality_df['duplicate_ts'] > 0
flags['FLAG_has_large_gaps']         = site_quality_df['n_large_gaps'] > 0
flags['FLAG_csip_mostly_null']       = site_quality_df['csip_opmod_exp_lim_null_pct'] > 50
flags['FLAG_solar_limit_mostly_null']= site_quality_df['solar_real_power_limit_null_pct'] > 80
flags['FLAG_negative_solar']         = site_quality_df['pct_negative_solar'] > 1
flags['FLAG_voltwatt_exposure']   = site_quality_df['pct_above_VW1'] > 5   # >5% time in VW zone
flags['FLAG_over_voltage']        = site_quality_df['pct_above_VW2'] > 1   # >1% time above VW2
flags['FLAG_solar_energy_null']      = site_quality_df['solar_to_battery_energy_null_pct'] > 50

flags['n_flags'] = flags[[c for c in flags.columns if c.startswith('FLAG_')]].sum(axis=1)

print("=== Quality flag summary ===")
print(flags[[c for c in flags.columns if c.startswith('FLAG_')]].sum().rename('sites_flagged'))

print("\n=== Sites with 3+ flags ===")
print(flags[flags['n_flags'] >= 3].sort_values('n_flags', ascending=False).to_string())

In [ ]:
# Export
# OUTPUT_DIR = Path('.')  # change as needed

# site_quality_df.to_csv(OUTPUT_DIR / 'site_quality_summary.csv')
# flags.to_csv(OUTPUT_DIR / 'site_quality_flags.csv')

# print("Exported:")
# print("  site_quality_summary.csv  — full numeric quality metrics per site")
# print("  site_quality_flags.csv    — pass/fail flags per site")

# 8. Testing

In [ ]:
all_data[all_data['pseudonym'] == 'SN005'].plot(x='timestamp', y='solar_to_battery_energy', figsize=(24,6))

In [ ]:
all_data[all_data['pseudonym'] == 'SN005'].plot(x='timestamp', y='solar_to_load_energy', figsize=(24,6))

In [ ]:
single_month = all_data[(all_data['timestamp'].dt.year == 2025) &
                        (all_data['timestamp'].dt.month == 11)]

single_month = (all_data
                .set_index('timestamp')
                .loc['2025-11']          # all rows in November 2025
                .reset_index())

single_month[single_month['pseudonym'] == 'SN005'][['timestamp', 'solar_to_load_energy', 'solar_to_battery_energy']].set_index('timestamp').plot(figsize=(24,6))

In [ ]:
# December 2025, first week
single_month = all_data[
    (all_data['timestamp'] >= '2025-12-04') &
    (all_data['timestamp'] < '2025-12-05')
].copy()

In [ ]:
single_month[single_month['pseudonym'] == 'SN005'][['timestamp', 'battery_inverter_real_power', 'AC_voltage']].set_index('timestamp').plot(
    figsize=(24, 6),
    secondary_y='AC_voltage'
)